In [1]:
import sys
import os

import bigframes.pandas as bpd
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ndcg_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils import shuffle

import optuna
from optuna.integration import XGBoostPruningCallback
import xgboost as xgb 

import gcsfs
import gc

import json
import pickle

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from config.config import BUCKET_NAME, source_loc, output_loc, config_loc

sys.path.insert(0, source_loc)
from utilities.utility_functions import split_data, select_features, get_bq_data_sample, pickle_and_stream_to_gcs

import warnings
warnings.filterwarnings('ignore')

In [2]:
best_trial_score = -1.0
best_trees_per_fold = [] #empty list to hold # of trees for each fold of best optuna trial

##### Read data into big frame and create sample

In [3]:
df_sample = get_bq_data_sample()

Full data shape: (18931851, 39)
Sample data shape: (1803821, 39)


##### Select features and split data

In [7]:
#Select only features an target column for training
df_features, feature_list, sort_list = select_features(df_sample)

In [8]:
#Perform split and only return train_val_df to save memory
train_val_df, _, _ = split_data(df_features)
train_val_df = train_val_df.reset_index(drop = True)

#Remove df_features from memory
del df_features
gc.collect()

216

##### Define objective for Bayesian search. Only performing light hyperparameter tuning due to size of data and later ensemble optimization

In [9]:
#Set n_estimators to 2000 for safe cap
#Set learning rate to 0.05 (half of default for smaller steps while also not going too low due to data size)

#Set the objective to binary:logistic for binary classificaiton

#Set evaluation metric to average precision (precision-recall AUC) due to class imbalance. ROC AUC is avoided due to the potential inflation
#of this metric due to true negatives. With average precision, True negatives are ignored and the focus is on minimizing false negatives and false positives.
#This aligns with the business case as we want to limit false negatives (predict re-orders correctly) while also minimizing the 
#amount of false negatives (predicting a reorder that was not re-ordered).

#Goal of this light hyperparameter tuning is to find optimal shape of trees rather trying to squeeze out fractional gains with a lower learning rate.

#reg lambda is looked at for L2 regularization due to few features combining for majority of importance (found in eda feature screening)
#Note: reg lambda penalizes the model for splitting on dominant features over and over again. It limits the importance of dominant features by preventing trees
#      from over-relying on these dominant features.

#Explore max depth in bayesian search instead of num_leaves (due to level_wise growth for xgboost instead of leafwise)
# Limit tree depth to 10 max due to data size and to prevent overfitting (avoid repetitive splitting on same features with only 14 features)

#Explore colsample_bytree (create diversity in trees for regularization/generalization)

#Utilize tree_method: 'hist' for performance due to data size. This utilizes histogram based binning to build decision tress instead of exact greedy splits.
#Thus instead of evaluating the gain of making a split at every single unique value for every feature at every node, the values for each feature are divided into bins
#and splits are evaluated only at the bin boundaries (which is significantly less computationally expensive)

def objective(trial):
    #Use global best_overall_score to track best score across all trials so best predictions can be saved locally
    global best_trial_score, best_trees_per_fold
    

    #Define trial hyperparameters
    params = {
            'objective': 'binary:logistic', 'eval_metric': 'aucpr', 
            'tree_method': 'hist', 'device': 'cuda', 'learning_rate': 0.05, 'n_estimators': 2000,
            'max_depth': trial.suggest_int('max_depth', 5, 10),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True)
            }

    #Create StratifiedGroupKFold instance
    sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

    #Create empty cv cores list to track cross validation scores
    cv_scores = []

    #Create array to track predictions
    trial_oof_preds = np.zeros(len(train_val_df))

    #Create list to track trees per fold
    trial_trees_per_fold = []

    for train_index, val_index in sgkf.split(train_val_df[feature_list], train_val_df['label_reordered'], groups=train_val_df['user_id']):
    
        print('Starting new fold...')

        #Create train and val data sets for current fold
        train_fold = train_val_df.iloc[train_index].copy()
        val_fold = train_val_df.iloc[val_index].copy()

        #Use the same method used for the rankers to track the original index. This ensures code consistency and acts as an insurance policy against index scrambling.
        val_fold['original_index'] = val_fold.index
    
        #create df for x_train, x_val, and numpy array for y_train, and y_val (Use arrays to remove index map and reduce memory overhead)
        x_train = train_fold[feature_list]
        y_train = train_fold['label_reordered'].to_numpy(dtype=np.int8)
        x_val = val_fold[feature_list]
        y_val = val_fold['label_reordered'].to_numpy(dtype=np.int8)

        print('x_train shape: ', x_train.shape)
        print('y_train shape: ', y_train.shape)
        print('x_val shape: ', x_val.shape)
        print('y_val shape: ', y_val.shape)


        #Create pruning callback to kill poor performing trials early
        pruning_callback = XGBoostPruningCallback(trial, 'validation_0-aucpr')

        #Create xgboost_classifier model instance & utilize params dictionary for hyperparameters
        #Utilize early stopping for regularization. Set to 50 rounds to balance with learning rate 0.05
        xgboost_classifier = xgb.XGBClassifier(early_stopping_rounds=50, callbacks=[pruning_callback], **params)


        #Fit the model
        #verbose = False to prevent flooding of print statements
        xgboost_classifier.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
        
        # Append validation precision recall auc to cv scores 
        cv_scores.append(max(xgboost_classifier.evals_result()['validation_0']['aucpr']))

        #Save predictions for this fold and map to original index. By the end of all folds there will be a prediction for all of train_val_df mapped back to the original index
        trial_oof_preds[val_fold['original_index']] = xgboost_classifier.predict_proba(x_val)[:, 1]

        #Append tree count to trial_trees_per_fold
        trial_trees_per_fold.append(xgboost_classifier.best_iteration) 

        # Remove from memory to free up RAM due to data size
        del train_fold, val_fold, x_train, x_val, y_train, y_val, xgboost_classifier
        gc.collect()

    avg_cv_score = np.mean(cv_scores)

    #Manual pruning to kill poor performing trails early
    #trial.report(avg_cv_score, step=trial.number)
    #if trial.should_prune():
     #   raise optuna.exceptions.TrialPruned()

    #If this is the best trial so far, save the oof predictions locally 
    if avg_cv_score > best_trial_score:
        best_trial_score = avg_cv_score
        best_trees_per_fold = trial_trees_per_fold
       
        oof_df = train_val_df[['user_id', 'anchor_order_number', 'label_reordered']].copy()
        oof_df['xgboost_classifier_pred'] = trial_oof_preds
        oof_df.to_parquet(os.path.join(output_loc, "xgboost_classifier_oof_preds.parquet"))

    # Remove from memory to free up RAM due to data size
    del trial_oof_preds
    gc.collect() 
            
    return avg_cv_score

##### Perform Bayesian Search

In [10]:
#Initialize study. Set it to maximize the objective
study = optuna.create_study(direction='maximize')

#Optimize utilizing objective function
#Set trials = 25 to enable the study to wrong long enough to find optimized parameters while also balancing data size
study.optimize(objective, n_trials=3)

#Get optimal trees per fold
optimal_trees = int(np.mean(best_trees_per_fold))

#Create params and trees dictionary
params = {
        "best_params": study.best_params,
        "optimal_trees": optimal_trees,
        "best_trees_per_fold": best_trees_per_fold
        }

[I 2026-08-02 20:06:52,663] A new study created in memory with name: no-name-4931af87-131a-4b94-9d3e-a7ecebe4ef0c


Starting new fold...
x_train shape:  (955982, 15)
y_train shape:  (955982,)
x_val shape:  (478034, 15)
y_val shape:  (478034,)
Starting new fold...
x_train shape:  (956088, 15)
y_train shape:  (956088,)
x_val shape:  (477928, 15)
y_val shape:  (477928,)
Starting new fold...
x_train shape:  (955962, 15)
y_train shape:  (955962,)
x_val shape:  (478054, 15)
y_val shape:  (478054,)


[I 2026-08-02 20:07:01,552] Trial 0 finished with value: 0.419245320975459 and parameters: {'max_depth': 8, 'colsample_bytree': 0.6708593898983831, 'reg_lambda': 7.272061217640665}. Best is trial 0 with value: 0.419245320975459.


Starting new fold...
x_train shape:  (955982, 15)
y_train shape:  (955982,)
x_val shape:  (478034, 15)
y_val shape:  (478034,)
Starting new fold...
x_train shape:  (956088, 15)
y_train shape:  (956088,)
x_val shape:  (477928, 15)
y_val shape:  (477928,)
Starting new fold...
x_train shape:  (955962, 15)
y_train shape:  (955962,)
x_val shape:  (478054, 15)
y_val shape:  (478054,)


[I 2026-08-02 20:07:09,507] Trial 1 finished with value: 0.41902949766491754 and parameters: {'max_depth': 8, 'colsample_bytree': 0.8253981431453555, 'reg_lambda': 0.43836554709301184}. Best is trial 0 with value: 0.419245320975459.


Starting new fold...
x_train shape:  (955982, 15)
y_train shape:  (955982,)
x_val shape:  (478034, 15)
y_val shape:  (478034,)
Starting new fold...
x_train shape:  (956088, 15)
y_train shape:  (956088,)
x_val shape:  (477928, 15)
y_val shape:  (477928,)
Starting new fold...
x_train shape:  (955962, 15)
y_train shape:  (955962,)
x_val shape:  (478054, 15)
y_val shape:  (478054,)


[I 2026-08-02 20:07:18,045] Trial 2 finished with value: 0.4198580348205534 and parameters: {'max_depth': 6, 'colsample_bytree': 0.6605797307246745, 'reg_lambda': 7.884905296069179}. Best is trial 2 with value: 0.4198580348205534.


In [11]:
# Save the best hyperparameters locally
param_file_path = os.path.join(config_loc, "xgboost_classifier_best_params.json")
with open(param_file_path, "w") as f:
    json.dump(params, f)

params

{'best_params': {'max_depth': 6,
  'colsample_bytree': 0.6605797307246745,
  'reg_lambda': 7.884905296069179},
 'optimal_trees': 158,
 'best_trees_per_fold': [118, 142, 215]}

##### Train model on full data and save to cloud storage bucket

In [12]:
#Train model on full data
train_full = train_val_df
x_full = train_full[feature_list].copy()
y_full = train_full['label_reordered'].to_numpy(dtype=np.int8)

In [13]:
#Create xgboost classifier instance (use optimal trees and best_params found from study)
xgboost_classifier_final = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr', tree_method='hist', device='cuda', learning_rate=0.05, n_estimators=optimal_trees, **study.best_params)

#Fit the model
xgboost_classifier_final.fit(x_full, y_full)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.6605797307246745
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[typing.Callable, str]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets impor

In [16]:
#Save final xgboost classifier to storage
pickle_and_stream_to_gcs(xgboost_classifier_final, BUCKET_NAME, 'xgboost_classifier_base_model.pkl')

🎉 Success! xgboost_classifier_base_model.pkl successfully streamed to Cloud Storage.


True